# External validation (VARHA) — test the HUS model on the external cohort

In [ ]:
# --- standard library ---
import json
import os
from collections import Counter
from fractions import Fraction

# --- numeric / data ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import beta as beta_dist, beta

# --- plotting ---
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import pyplot
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter

# --- modelling ---
import xgboost as xgb
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    accuracy_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    GroupShuffleSplit,
    GroupKFold,
    GridSearchCV,
)
from xgboost import XGBClassifier

# --- notebook ---
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display

# --- other ---
import openpyxl

InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.max_columns', None)

DCA_BLUE, DCA_GRAY, DCA_INK, DCA_BAND = "#0072B2", "#7F7F7F", "#222222", "#E8EDF2"
DCA_MARK, DCA_OLD, DCA_CV = "#D55E00", "#9AA6B2", "#3F8F4F"


In [ ]:
# Create a truncated version of the Blues colormap
colourmap = mcolors.LinearSegmentedColormap.from_list(
    'Blues_truncated',
    plt.cm.Blues(np.linspace(0.1, 1.0, 256))
)

In [ ]:
SEED = 42
dfnames = ['NOimputation']
i = 0
#colourmap = 'Blues'

nof_features = [7]
thresholds = [0.20, 0.24]
THRESHOLD = thresholds[i]
DATA_SOURCE = dfnames[i]
PLOT_PATH = f'/path/to/plots/{DATA_SOURCE}/'
DATA_PATH = f"/path/to/data/"
PLOT_DATA_PATH = "/path/to/plot_data/"
print("Data source:", DATA_SOURCE)
print("Threshold:", THRESHOLD)

## Load model

In [ ]:
# 1) load model
model = XGBClassifier()
#model.load_model(DATA_PATH + f"simple_models/simple_xgb_model.json")
model.load_model(DATA_PATH + f"simple_models/simple_xgb_model.json")

# 2) Load feature names from JSON
with open(DATA_PATH + f"simple_models/simple_features.json", "r") as f:
    feature_names = json.load(f)  # e.g. ["feature1", "feature2", ...]


## Load test data

**Which figures this notebook makes depends on `EXCLUDED` below:**
- `EXCLUDED = False` (full external cohort): **Figure 6**, **Supplementary Figure 10** and the external column of **Supplementary Table 4**
- `EXCLUDED = True` (patients without antibiotic exposure): **Supplementary Figure 12**

In [ ]:
df = pd.read_csv(f"{DATA_PATH}model_data_{DATA_SOURCE}.csv")
df = df[df['fn_day']==0]
print(f"{DATA_PATH}model_data_{DATA_SOURCE}.csv")

# ---------------------------------------------------------
# use this if you want to test with AB excluded cohort
EXCLUDED = True

if EXCLUDED:    
    PLOT_PATH = f'/path/to/plots/{DATA_SOURCE}/excluded/'
    PLOT_DATA_PATH = "/path/to/plot_data/ab_excluded/"
    excluded_cohort = pd.read_excel('/path/to/tulokset/potilaat_ja_paivat_ilman_ab_hoitoa.xlsx')[['henkilotunnus', 'naytteenotto_hetki']]

    keys = ["henkilotunnus", "naytteenotto_hetki"]

    idx = pd.MultiIndex.from_frame(excluded_cohort[keys])
    df = df[df.set_index(keys).index.isin(idx)]
    
# --------------------------------------------------------

X_test = df[list(set(np.array(feature_names)) & set(df.columns.tolist()))]
X_test.to_csv(DATA_PATH+'x_test.csv')

missing_columns = list(set(np.array(feature_names)) - set(df.columns.tolist()))
print(missing_columns)

X_test[missing_columns] = np.nan
X_test = X_test.reindex(columns=np.array(feature_names), fill_value=0)

y_test = df['infektion_binary']
group_test = df['henkilotunnus']


In [ ]:
print('Number of induction datapoints:', len(df[df['sykli_IND']==1]))
print('Number of consolidation datapoints:', len(df[df['sykli_IND']==0]))
print()
print('Number of pateints in external test set', len(df['henkilotunnus'].drop_duplicates()))
print()
print('Number of treatment cycles in external test set:', len(df.groupby(['henkilotunnus', 'cycle_number']).size()))
print()
print('Number of datapoints in external test set:', len(df))
print()
print("Number of FN in external testing set", int(df.reset_index()['infektion_binary'].sum()))

## Functions

In [ ]:
# ---------------------------------------------------------------------------
# Source-data export for figures (journal request)
# ---------------------------------------------------------------------------

_IMG_EXT = {".png", ".pdf", ".svg", ".eps", ".jpg", ".jpeg", ".tif", ".tiff"}


def _plot_data_stem(name):
    """Figure stem from a save_path. Strips only real image extensions, so
    'test_cm_0_5' and 'test_cm_0.5.png' both come out right."""
    base = os.path.basename(str(name).rstrip("/"))
    root, ext = os.path.splitext(base)
    return root if ext.lower() in _IMG_EXT else base


def _slugify(text):
    out = "".join(c if (c.isalnum() or c in "-_") else "_" for c in str(text))
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_").lower()[:80] or "figure"


def write_plot_data(name, tables, out_dir=None, note="", manifest=True,
                    verbose=True):
    """Write the source data behind one figure as CSV.

    name   : figure stem -- pass the same save_path you gave savefig.
    tables : {part_name: DataFrame}
             one table  -> PLOT_DATA_PATH/<stem>.csv
             several    -> PLOT_DATA_PATH/<stem>/<stem>__<part>.csv
    note   : free text recorded in the manifest (threshold, cohort, ...).

    Returns the list of paths written.
    """
    out_dir = out_dir or PLOT_DATA_PATH
    stem = _plot_data_stem(name)
    tables = {k: df for k, df in tables.items() if df is not None and len(df)}
    if not tables:
        if verbose:
            print(f"[plot_data] {stem}: nothing to write")
        return []

    target = out_dir if len(tables) == 1 else os.path.join(out_dir, stem)
    os.makedirs(target, exist_ok=True)

    paths, rows = [], []
    for part, df in tables.items():
        fname = f"{stem}.csv" if len(tables) == 1 else f"{stem}__{part}.csv"
        path = os.path.join(target, fname)
        df.to_csv(path, index=False)
        paths.append(path)
        rows.append({"figure": stem,
                     "part": "" if len(tables) == 1 else part,
                     "file": os.path.relpath(path, out_dir),
                     "n_rows": len(df),
                     "columns": "; ".join(map(str, df.columns)),
                     "note": note})

    if manifest:
        idx = os.path.join(out_dir, "plot_data_index.csv")
        new = pd.DataFrame(rows)
        if os.path.exists(idx):
            old = pd.read_csv(idx, keep_default_na=False)
            old = old[old["figure"].astype(str) != stem]   # rerun replaces
            new = pd.concat([old, new], ignore_index=True)
        new.to_csv(idx, index=False)

    if verbose:
        print(f"[plot_data] {stem}: " +
              ", ".join(os.path.relpath(p, out_dir) for p in paths))
    return paths

In [ ]:
def plot_confusion_matrix_with_percentages(
    y_true,
    y_pred,
    labels=None,
    percent_type="row",          # 'row', 'column', or 'all'
    title=None,
    cmap="viridis",
    threshold=None,
    save_path=None,
    figsize=None,
    text_color=None,
    plot_data=False,             # <-- NEW: write the source-data CSV
    plot_data_path=None,         # <-- NEW: defaults to PLOT_DATA_PATH
    plot_data_name=None,         # <-- NEW: defaults to save_path / title
):
    from sklearn.metrics import confusion_matrix
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if labels is None:
        # must match the label order confusion_matrix actually used
        labels = np.unique(np.concatenate([np.asarray(y_true).ravel(),
                                           np.asarray(y_pred).ravel()]))

    # Calculate percentages
    if percent_type == "row":
        sums = cm.sum(axis=1, keepdims=True)
        perc = cm / np.where(sums == 0, 1, sums) * 100
        perc_label = "Row %"
    elif percent_type == "column":
        sums = cm.sum(axis=0, keepdims=True)
        perc = cm / np.where(sums == 0, 1, sums) * 100
        perc_label = "Col %"
    elif percent_type == "all":
        total = cm.sum()
        perc = (cm / total * 100) if total != 0 else np.zeros_like(cm, dtype=float)
        perc_label = "All %"
    else:
        raise ValueError("percent_type must be 'row', 'column', or 'all'")

    # Annotation with count and percentage
    annot = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]}\n({perc[i, j]:.0f}%)"

    fig, ax = plt.subplots(figsize=figsize)

    # ✅ Heatmap uses PERCENTAGES so colors + colorbar are %
    im = ax.imshow(perc, cmap=cmap, vmin=0, vmax=100)
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label("Percent (%)", fontsize=14)

    # Show numbers
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if text_color is None:
                # background is % (0..100), so use 50 as midpoint
                cell_color = "black" if perc[i, j] > 50 else "white"
            else:
                cell_color = text_color

            ax.text(
                j, i, annot[i, j],
                ha="center", va="center",
                fontsize=13,
                color=cell_color
            )

    ax.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted label",
        ylabel="True label",
    )
    ax.set_xlabel("Predicted label", fontsize=14)
    ax.set_ylabel("True label", fontsize=14)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)

    full_title = title if title else f"Confusion Matrix with {perc_label}"
    ax.set_title(full_title, fontsize=14)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    # ---- source data for the journal --------------------------------------
    data_paths = []
    if plot_data:
        rows = []
        for i, t in enumerate(labels):
            for j, p in enumerate(labels):
                rows.append({
                    "true_label": t,
                    "predicted_label": p,
                    "count": int(cm[i, j]),
                    "percent": float(perc[i, j]),
                    "percent_basis": percent_type,
                })
        note = f"percent_type={percent_type}"
        if threshold is not None:
            note += f"; threshold={threshold}"
        data_paths = write_plot_data(
            plot_data_name or save_path or _slugify(full_title),
            {"confusion_matrix": pd.DataFrame(rows)},
            out_dir=plot_data_path,
            note=note,
        )

    plt.show()
    return data_paths

In [ ]:
def custom_score(y_true, y_pred, group_ids, threshold, eval=False):
    # Binary prediction
    y_bin = (y_pred >= threshold).astype(int)
    print(f"f1 before mod: {f1_score(y_true, y_bin)}")
    # Create working DataFrame
    df = pd.DataFrame({
        'henkilotunnus': group_ids,
        'true_label': y_true,
        'predicted_label': y_bin
    })

    df['misclassified'] = (df['true_label'] != df['predicted_label']).astype(int)
    df['misclassified_proximity'] = 0

    # Set proximity rule
    for name, group in df.groupby('henkilotunnus'):
        for i in range(len(group) - 1):
            row = group.iloc[i]
            next_row = group.iloc[i + 1]

            if (
                row['misclassified'] == 1
                and row['true_label'] == 0
                and row['predicted_label'] == 1
                and next_row['true_label'] == 1
            ):
                df.loc[group.index[i], 'misclassified_proximity'] = 1
 
    # Adjust predictions
    adjusted_pred = df['predicted_label'].copy()
    mask = (df['misclassified'] == 1) & (df['misclassified_proximity'] == 1)
    adjusted_pred[mask] = df['true_label'][mask]

    if eval:
        return df, f1_score(df['true_label'], adjusted_pred), adjusted_pred
    else:
        return f1_score(df['true_label'], adjusted_pred)

In [ ]:
def calc_pnv(y_true, y_pred):
    """
    Compute PNV (Negative Predictive Value) for binary data.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    TN = np.sum((y_pred == 0) & (y_true == 0))  # true negatives
    FN = np.sum((y_pred == 0) & (y_true == 1))  # false negatives

    if TN + FN == 0:
        return np.nan  # no predicted negatives → PNV undefined

    return TN / (TN + FN)

## Predict

In [ ]:
def custom_f1(y_true, y_pred, multi=3):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fp_adj = fp / multi
    precision = tp / (tp + fp_adj) if (tp + fp_adj) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

In [ ]:
print('\nTest set:')

y_pred_test = model.predict_proba(X_test)[:, 1]
y_pred_thresh = (y_pred_test >= THRESHOLD).astype(int)

kappa = cohen_kappa_score(y_test, y_pred_thresh)
recall = recall_score(y_test, y_pred_thresh)
precision = precision_score(y_test, y_pred_thresh)
f1 = f1_score(y_test, y_pred_thresh)
accuracy = accuracy_score(y_test, y_pred_thresh)
customf1 = custom_f1(y_test, y_pred_thresh)
spec = recall_score(y_test, y_pred_thresh, pos_label=0)

metrics_df = pd.DataFrame({'data':[f'simple_test_{THRESHOLD}'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 'specificity': [spec], 'PNV':[calc_pnv(y_test, y_pred_thresh)]})

print('Kappa:', kappa)
print("Recall:", recall)
print("Precision:", precision)
print("F1 score:", f1)
print("Custom F1:", customf1)

**Figure 6A** -- Confusion matrix, external validation cohort (threshold 0.2)

*With `EXCLUDED = True`:* **Supplementary Figure 12A**

In [ ]:

y_prob_test = model.predict_proba(X_test)[:, 1]

# confusion matrix with changed threshold
y_pred_thresh = (y_prob_test >= THRESHOLD).astype(int)
plot_confusion_matrix_with_percentages(
    y_test, y_pred_thresh, 
    percent_type='row', 
    threshold=THRESHOLD,
    save_path = PLOT_PATH+'test_cm_'+str(THRESHOLD).replace('.', '_'),  # <-- save figure here
    title=f'External validation set',
    #cmap='coolwarm',
    cmap = colourmap,
    text_color='white',
    plot_data=True,
)

kappa = cohen_kappa_score(y_test, y_pred_thresh)
recall = recall_score(y_test, y_pred_thresh)
precision = precision_score(y_test, y_pred_thresh)
f1 = f1_score(y_test, y_pred_thresh)
accuracy = accuracy_score(y_test, y_pred_thresh)
customf1 = custom_f1(y_test, y_pred_thresh)

print(f'Kappa       : {kappa:.4f}') # Is my model better than random, and by how much?
print(f"Recall      : {recall:.4f}")
print(f"Precision   : {precision:.4f}")
print(f"F1 score    : {f1:.4f}")
print(f"Custom F1   : {customf1:.4f}")

### AUCROC and PRAUC

**Figure 6B** -- AUROC with 95% CI, external validation cohort

**Supplementary Figure 10C** -- AUPRC with 95% CI, external validation cohort

*With `EXCLUDED = True`:* **Supplementary Figure 12B** (AUROC) and **12D** (AUPRC)

In [ ]:
def plot_curve_ci(y_true, y_prob, kind="roc", groups=None, n_boot=1000,
                  seed=42, alpha=0.05, n_grid=201, ax=None, title=None,
                  figsize=(7.0, 7.0),
                  plot_data=False, plot_data_path=None, plot_data_name=None):
    """ROC or precision-recall curve with a bootstrap 95% band and AUC (95% CI)."""
    y = np.asarray(y_true).astype(int); p = np.asarray(y_prob, float)
    grid = np.linspace(0, 1, n_grid)

    def _curve(yy, pp):
        if kind == "roc":
            fpr, tpr, _ = roc_curve(yy, pp)
            return np.interp(grid, fpr, tpr), roc_auc_score(yy, pp)
        pr, rc, _ = precision_recall_curve(yy, pp)
        return np.interp(grid, rc[::-1], pr[::-1]), average_precision_score(yy, pp)

    y_pt, auc_pt = _curve(y, p)

    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    _, inv = np.unique(g, return_inverse=True)
    idx = [np.flatnonzero(inv == k) for k in range(inv.max() + 1)]

    rng = np.random.default_rng(seed)
    curves = np.full((n_boot, n_grid), np.nan)
    aucs = np.full(n_boot, np.nan)
    for b in range(n_boot):
        s = rng.integers(0, len(idx), len(idx))
        ii = np.concatenate([idx[j] for j in s])
        yb, pb = y[ii], p[ii]
        if yb.min() == yb.max():
            continue
        curves[b], aucs[b] = _curve(yb, pb)

    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    band_lo = np.nanpercentile(curves, lo_q, axis=0)
    band_hi = np.nanpercentile(curves, hi_q, axis=0)
    a_lo, a_hi = np.nanpercentile(aucs, lo_q), np.nanpercentile(aucs, hi_q)

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    name = "AUROC" if kind == "roc" else "AUPRC"
    ax.fill_between(grid, band_lo, band_hi, color=DCA_BLUE, alpha=0.20, lw=0, zorder=1)
    ax.plot(grid, y_pt, color=DCA_BLUE, lw=2, zorder=3)

    if kind == "roc":
        ax.plot([0, 1], [0, 1], color=DCA_GRAY, ls="--", lw=1, zorder=2)
        ax.set_xlabel("False positive rate", fontsize=14)
        ax.set_ylabel("True positive rate", fontsize=14)
        curve_lbl, loc = "ROC curve", "lower right"
    else:
        base = y.mean()
        ax.axhline(base, color=DCA_GRAY, ls="--", lw=1, zorder=2)
        ax.set_xlabel("Recall", fontsize=14)
        ax.set_ylabel("Precision", fontsize=14)
        curve_lbl, loc = "Precision–recall curve", "upper right"

    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.1)

    _line = Line2D([], [], color=DCA_BLUE, lw=2)
    _band = Patch(facecolor=DCA_BLUE, alpha=0.20, lw=0)
    _txt  = Line2D([], [], ls="none", marker="none")
    _auc  = f"{name} = {auc_pt:.2f} (95% CI {a_lo:.2f}–{a_hi:.2f})"
    ax.legend([_line, _band, _txt],
              [curve_lbl, "Curve – 95% CI", _auc],
              fontsize=11, loc=loc, frameon=False)

    ax.set_title(title or ("ROC curve" if kind == "roc" else "Precision-recall curve"),
                 fontsize=16)
    ax.set_box_aspect(1)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    # ---- source data for the journal --------------------------------------
    if plot_data:
        import pandas as pd
        if kind == "roc":
            curve = pd.DataFrame({"false_positive_rate": grid,
                                  "true_positive_rate": y_pt,
                                  "tpr_ci_lower": band_lo,
                                  "tpr_ci_upper": band_hi})
            baseline = np.nan
        else:
            curve = pd.DataFrame({"recall": grid,
                                  "precision": y_pt,
                                  "precision_ci_lower": band_lo,
                                  "precision_ci_upper": band_hi})
            baseline = float(base)
        summary = pd.DataFrame([{
            "metric": name,
            "value": float(auc_pt),
            "ci_lower": float(a_lo),
            "ci_upper": float(a_hi),
            "ci_level": 1 - alpha,
            "n_bootstrap": int(n_boot),
            "n_bootstrap_used": int(np.isfinite(aucs).sum()),
            "bootstrap_unit": "observation" if groups is None else "group",
            "seed": int(seed),
            "n_total": int(y.size),
            "n_positive": int(y.sum()),
            "prevalence": float(y.mean()),
            "reference_line": baseline,
        }])
        write_plot_data(
            plot_data_name or _slugify(ax.get_title()) or f"{kind}_curve",
            {"curve": curve, "summary": summary},
            out_dir=plot_data_path,
            note=f"{name}={auc_pt:.4f} (95% CI {a_lo:.4f}-{a_hi:.4f}); "
                 f"n_boot={n_boot}; grid={n_grid} points",
        )

    print(f"{name} {auc_pt:.2f} (95% CI {a_lo:.2f}-{a_hi:.2f})")
    return fig, (auc_pt, a_lo, a_hi)


fig, roc_ci  = plot_curve_ci(y_test, y_prob_test, kind="roc", groups=None, plot_data=True, plot_data_name='test_ROC_curve_CI')
fig.savefig(PLOT_PATH + 'test_ROC_curve_CI', dpi=300, bbox_inches='tight')
plt.show()

fig, prc_ci = plot_curve_ci(y_test, y_prob_test, kind="pr", groups=None, plot_data=True, plot_data_name='test_RP_curve_CI')
fig.savefig(PLOT_PATH + 'test_RP_curve_CI', dpi=300, bbox_inches='tight')
plt.show()

### Metrics with 95%CI

In [ ]:
def _row_metrics(y, yh):
    return {
        "F1_score":    f1_score(y, yh, zero_division=0),
        "kappa":       cohen_kappa_score(y, yh),
        "recall":      recall_score(y, yh, zero_division=0),
        "precision":   precision_score(y, yh, zero_division=0),
        "accuracy":    accuracy_score(y, yh),
        "customf1":    custom_f1(y, yh),
        "specificity": recall_score(y, yh, pos_label=0, zero_division=0),
        "PNV":         calc_pnv(y, yh),
    }

def add_metrics_column(table, y_true, y_prob, thr, name, groups=None, extra=None,
                       n_boot=2000, seed=42, alpha=0.05, add_auc=True):
    """Append one run to the metrics table as the column group (name, est/lo/hi).
       table=None starts a new table. Re-using a name replaces that run."""
    y = np.asarray(y_true).astype(int); p = np.asarray(y_prob, float)

    def _all(yy, pp):
        m = _row_metrics(yy, (pp >= thr).astype(int))
        if add_auc:
            m = {"AUROC": roc_auc_score(yy, pp),
                 "AUPRC": average_precision_score(yy, pp), **m}
        if extra:
            for k, fn_ in extra.items():
                m[k] = fn_(yy, (pp >= thr).astype(int))
        return m

    point = _all(y, p)
    keys = list(point)

    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    _, inv = np.unique(g, return_inverse=True)
    idx = [np.flatnonzero(inv == k) for k in range(inv.max() + 1)]

    rng = np.random.default_rng(seed)
    boots = np.full((n_boot, len(keys)), np.nan)
    for b in range(n_boot):
        s = rng.integers(0, len(idx), len(idx))
        ii = np.concatenate([idx[j] for j in s])
        yb, pb = y[ii], p[ii]
        if yb.min() == yb.max():
            continue
        m = _all(yb, pb)
        boots[b] = [m[k] for k in keys]

    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    col = pd.DataFrame(
        {(name, "est"): [point[k] for k in keys],
         (name, "lo"):  np.nanpercentile(boots, lo_q, axis=0),
         (name, "hi"):  np.nanpercentile(boots, hi_q, axis=0)},
        index=pd.Index(keys, name="metric"))
    col.columns = pd.MultiIndex.from_tuples(col.columns, names=["run", "stat"])

    if table is None:
        return col
    if name in table.columns.get_level_values("run"):
        table = table.drop(columns=name, level="run")
    order = list(table.index) + [k for k in col.index if k not in table.index]
    return table.join(col, how="outer").reindex(order)


def fmt_table(df, decimals=2):
    """Collapse each run's est/lo/hi into one 'est (lo–hi)' column."""
    out = pd.DataFrame(index=df.index)
    for r in df.columns.get_level_values("run").unique():
        e, lo, hi = df[(r, "est")], df[(r, "lo")], df[(r, "hi")]
        out[r] = [f"{a:.{decimals}f} ({b:.{decimals}f}–{c:.{decimals}f})"
                  if pd.notna(a) else "" for a, b, c in zip(e, lo, hi)]
    return out

In [ ]:
metrics_df_ci = add_metrics_column(None, y_test, y_prob_test,  THRESHOLD, "External")
fmt_table(metrics_df_ci)

**Supplementary Table 4** -- Extended prediction window, external validation cohort (false positives the day before FN counted as true positives)

In [ ]:
y_prob_test = model.predict_proba(X_test)[:, 1]
#y_pred_proba = best_model1.predict_proba(X_test)[:, 1]
#THRESHOLD = 0.1
df, score, y_pred_mod = custom_score(y_test, y_prob_test, group_test, threshold=THRESHOLD, eval=True)

# 👉 3. Evaluation metrics
f1 = f1_score(y_test, y_pred_mod)
recall = recall_score(y_test, y_pred_mod)
accuracy = accuracy_score(y_test, y_pred_mod)
precision = precision_score(y_test, y_pred_mod)
kappa = cohen_kappa_score(y_test, y_pred_mod)
customf1 = custom_f1(y_test, y_pred_mod)
spec = recall_score(y_test, y_pred_mod, pos_label=0)

print(f"Kappa       : {kappa:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
print(f"Custom F1   : {customf1:.4f}")

# 👉 4. Confusion matrix
plot_confusion_matrix_with_percentages(
    y_test, y_pred_mod, 
    percent_type='row', 
    threshold=THRESHOLD,
    save_path = PLOT_PATH+'test_set_cm_correction'+str(THRESHOLD).replace('.', '_'), # <-- save figure here
    title='External validation set (corrected)',
    cmap=colourmap,
    text_color='white',
    plot_data=True
)

row = pd.DataFrame({'data':[f'simple_test_corrected_{THRESHOLD}'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 
                    'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 'specificity': [spec], 'PNV':[calc_pnv(y_test, y_pred_mod)]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)
metrics_df

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "External corrected")
fmt_table(metrics_df_ci)



In [ ]:
def move_column_inplace(df, col, pos):
    col = df.pop(col)
    df.insert(pos, col.name, col)

In [ ]:
eval_df = pd.read_csv(DATA_PATH+f"/model_data_{DATA_SOURCE}.csv").drop(columns=['Unnamed: 0'])
if EXCLUDED:
    excluded_cohort = pd.read_excel('/path/to/tulokset/potilaat_ja_paivat_ilman_ab_hoitoa.xlsx')[['henkilotunnus', 'naytteenotto_hetki']]

    keys = ["henkilotunnus", "naytteenotto_hetki"]

    idx = pd.MultiIndex.from_frame(excluded_cohort[keys])
    eval_df = eval_df[eval_df.set_index(keys).index.isin(idx)]
    
eval_df = eval_df[eval_df['fn_day']==0]
eval_df['pred_modified_y'] = y_pred_mod
eval_df['original_y'] = y_test
eval_df['pred_proba'] = y_prob_test
eval_df['y_pred'] = y_pred_thresh
move_column_inplace(eval_df, 'pred_proba', 1)
move_column_inplace(eval_df, 'pred_modified_y', 1)
move_column_inplace(eval_df, 'original_y', 1)
move_column_inplace(eval_df, 'y_pred', 1)
move_column_inplace(eval_df, 'naytteenotto_hetki', 5)


missing_columns = list(set(np.array(feature_names)) - set(eval_df.columns.tolist()))

eval_df[missing_columns] = np.nan
eval_df.rename(columns={'original_y':'y_original'}).to_csv(DATA_PATH + 'eval_df.csv')

### DCA 

In [ ]:
# --- standardise to the column names used in all later steps ---
dca_df = (eval_df
          .rename(columns={"henkilotunnus": "patient_id",
                           "pred_proba":    "p",
                           "original_y":    "y"})
          [["patient_id", "y", "p"]]
          .copy())
dca_df["y"] = dca_df["y"].astype(int)
dca_df["p"] = dca_df["p"].astype(float)

# --- Step 2: the counts that go in the manuscript ---
n_patients = dca_df["patient_id"].nunique()
n_days     = len(dca_df)
n_events   = int(dca_df["y"].sum())
prevalence = dca_df["y"].mean()

days_per_pat = dca_df.groupby("patient_id").size()
ev_per_pat   = dca_df.groupby("patient_id")["y"].sum()

print(f"Patients                      : {n_patients}")
print(f"Patient-days                  : {n_days}")
print(f"FN-positive patient-days      : {n_events}")
print(f"Prevalence per patient-day    : {prevalence:.4f}  ({prevalence*100:.2f} %)")
print(f"Patients with >=1 FN day      : {(ev_per_pat > 0).sum()} "
      f"({(ev_per_pat > 0).mean()*100:.1f} % of patients)")
print(f"Days per patient, median (IQR): {days_per_pat.median():.0f} "
      f"({days_per_pat.quantile(.25):.0f}-{days_per_pat.quantile(.75):.0f}), "
      f"range {days_per_pat.min()}-{days_per_pat.max()}")

# --- sanity checks: catch data problems before they become figure problems ---
print("\n--- sanity checks ---")
print("missing values       :", dca_df.isna().sum().to_dict())
print("y values present     :", sorted(dca_df["y"].unique()))
print("p range              :", f"{dca_df['p'].min():.4f} - {dca_df['p'].max():.4f}")
print("p outside [0,1]      :", int(((dca_df["p"] < 0) | (dca_df["p"] > 1)).sum()))
print("exact duplicate rows :", int(dca_df.duplicated().sum()))
print("mean predicted risk  :", f"{dca_df['p'].mean():.4f}", "| observed:", f"{prevalence:.4f}")

In [ ]:
def dca_logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return np.log(p / (1 - p))

_y  = dca_df["y"].to_numpy(int)
_p  = dca_df["p"].to_numpy(float)
_lp = dca_logit(_p)
_grp = dca_df["patient_id"].to_numpy()

# --- calibration intercept & slope, cluster-robust SEs by patient ---
try:
    import statsmodels.api as sm
    _fit = sm.GLM(_y, sm.add_constant(_lp), family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": _grp})
    dca_cal_intercept, dca_cal_slope = _fit.params
    _ci = _fit.conf_int()
    _int_lo, _int_hi     = _ci[0]
    _slope_lo, _slope_hi = _ci[1]

    # calibration-in-the-large: intercept with the slope pinned at 1
    _fit0 = sm.GLM(_y, np.ones((len(_y), 1)), family=sm.families.Binomial(),
                   offset=_lp).fit(cov_type="cluster", cov_kwds={"groups": _grp})
    _citl = _fit0.params[0]
    _citl_lo, _citl_hi = _fit0.conf_int()[0]
except ImportError:
    from sklearn.linear_model import LogisticRegression
    _lr = LogisticRegression(C=1e12, solver="lbfgs", max_iter=1000).fit(_lp.reshape(-1, 1), _y)
    dca_cal_intercept, dca_cal_slope = float(_lr.intercept_[0]), float(_lr.coef_[0][0])
    _int_lo = _int_hi = _slope_lo = _slope_hi = np.nan
    _citl = _citl_lo = _citl_hi = np.nan

_prev  = _y.mean()
_brier = float(np.mean((_p - _y) ** 2))
_brier_scaled = 1 - _brier / (_prev * (1 - _prev))

print(f"Calibration slope        : {dca_cal_slope:.3f}  [{_slope_lo:.3f}, {_slope_hi:.3f}]   (ideal 1)")
print(f"Calibration intercept    : {dca_cal_intercept:+.3f} [{_int_lo:+.3f}, {_int_hi:+.3f}]  (ideal 0)")
print(f"Calibration-in-the-large : {_citl:+.3f} [{_citl_lo:+.3f}, {_citl_hi:+.3f}]  (ideal 0)")
print(f"Brier score              : {_brier:.5f}")
print(f"Scaled Brier (skill)     : {_brier_scaled:.3f}   (0 = no better than predicting prevalence)")
print(f"Mean predicted {_p.mean():.4f} vs observed {_prev:.4f}  (ratio {_p.mean()/_prev:.2f})")

**Figure 6C** -- Calibration plot with prediction distribution, external validation cohort

*With `EXCLUDED = True`:* **Supplementary Figure 12C**

In [ ]:
def plot_calibration_full(df, n_bins=10, show_lowess=True, xmax=1.0, ymax=None,
                          ax=None, equal_aspect=False,
                          plot_data=False, plot_data_path=None,
                          plot_data_name=None):
    y = df["y"].to_numpy(int); p = df["p"].to_numpy(float)
    lp = dca_logit(p); grp = df["patient_id"].to_numpy()

    # the calibration curve itself supplies the smooth curve and its band,
    # with cluster-robust covariance so the band respects patient clustering
    fit = sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial()).fit(
            cov_type="cluster", cov_kwds={"groups": grp})
    gx = np.linspace(1e-4, xmax, 400)
    pr = fit.get_prediction(sm.add_constant(dca_logit(gx))).summary_frame(alpha=0.05)

    _b = pd.qcut(p, n_bins, labels=False, duplicates="drop")
    g = (pd.DataFrame({"p": p, "y": y, "bin": _b}).groupby("bin")
           .agg(mean_p=("p","mean"), obs=("y","mean"), n=("y","size"), ev=("y","sum")))
    _lo = np.nan_to_num(beta.ppf(0.025, g["ev"],     g["n"]-g["ev"]+1), nan=0.0)
    _hi = np.nan_to_num(beta.ppf(0.975, g["ev"] + 1, g["n"]-g["ev"]),   nan=1.0)
    _yerr = np.clip(np.vstack([g["obs"] - _lo, _hi - g["obs"]]), 0, None)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7.0, 7.6))
    else:
        fig = ax.figure

    ax.plot([0, xmax], [0, xmax], color=DCA_GRAY, ls="--", lw=1, zorder=2,
            label="Ideal calibration")
    ax.fill_between(gx, pr["mean_ci_lower"], pr["mean_ci_upper"],
                    color=DCA_BLUE, alpha=0.2, lw=0, zorder=1,
                    label="Curve - 95% confidence interval")
    ax.plot(gx, pr["mean"], color=DCA_BLUE, lw=2, zorder=4, label="Calibration curve")
    if show_lowess:
        try:
            from statsmodels.nonparametric.smoothers_lowess import lowess
            _sup = np.quantile(p, 0.995)        # don't draw beyond the data
            _s = lowess(y, p, frac=0.6, it=0, return_sorted=True)
            _m = _s[:, 0] <= min(xmax, _sup)
        except ImportError:
            pass

    ax.errorbar(g["mean_p"], g["obs"], yerr=_yerr, fmt="o", ms=5, lw=1, capsize=3,
                color=DCA_BLUE, zorder=5)
    _pt = Line2D([], [], color=DCA_BLUE, marker="o", ls="none", ms=5)
    _ci = ax.errorbar([np.nan], [np.nan], yerr=[[1.0], [1.0]], fmt="none",
                      ecolor=DCA_BLUE, elinewidth=1, capsize=3)

    ax.set_xlim(0, xmax); ax.set_ylim(0, ymax)
    ax.set_xlabel("Predicted probability", fontsize=14)
    ax.set_ylabel("Observed FN rate", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.1)

    _h, _l = ax.get_legend_handles_labels()
    _by = dict(zip(_l, _h))
    ax.legend([_pt, _ci, _by["Calibration curve"],
               _by["Curve - 95% confidence interval"], _by["Ideal calibration"]],
              ["Groups", "Group - SD", "Calibration curve",
               "Curve - 95% CI", "Ideal calibration"],
              fontsize=11, loc="upper left", frameon=False)

    ax.set_title("Calibration plot", fontsize=16)
    print(f"slope {fit.params[1]:.2f}, intercept {fit.params[0]:+.2f}")
    if equal_aspect:
        ax.set_aspect("equal", adjustable="box")
    else:
        ax.set_box_aspect(1)
    ax.spines[["top", "right"]].set_visible(False)

    _ax2 = ax.inset_axes([0, -0.36, 1, 0.20])
    _hist_n, _hist_edges, _ = _ax2.hist(p, bins=80, range=(0, xmax), color=DCA_OLD)
    _ax2.set_yticks([]); _ax2.set_xlim(0, xmax)
    _ax2.set_xlabel("Distribution of predictions", fontsize=10)
    _ax2.tick_params(labelsize=10)
    _ax2.spines[["top", "right", "left"]].set_visible(False)

    fig.tight_layout()

    # ---- source data for the journal --------------------------------------
    if plot_data:
        curve = pd.DataFrame({
            "predicted_probability": gx,
            "observed_rate": pr["mean"].to_numpy(),
            "observed_ci_lower": pr["mean_ci_lower"].to_numpy(),
            "observed_ci_upper": pr["mean_ci_upper"].to_numpy(),
        })
        bins = pd.DataFrame({
            "bin": g.index.to_numpy(),
            "mean_predicted_probability": g["mean_p"].to_numpy(),
            "observed_rate": g["obs"].to_numpy(),
            "observed_ci_lower": g["obs"].to_numpy() - _yerr[0],
            "observed_ci_upper": g["obs"].to_numpy() + _yerr[1],
            "n": g["n"].to_numpy(),
            "n_events": g["ev"].to_numpy(),
        })
        hist = pd.DataFrame({
            "bin_left": _hist_edges[:-1],
            "bin_right": _hist_edges[1:],
            "count": _hist_n.astype(int),
        })
        _b0, _b1 = np.asarray(fit.params, float)[:2]
        _ci95 = np.asarray(fit.conf_int(alpha=0.05), float)
        _se = np.asarray(fit.bse, float)
        summary = pd.DataFrame([{
            "calibration_intercept": _b0,
            "intercept_se": _se[0],
            "intercept_ci_lower": _ci95[0, 0],
            "intercept_ci_upper": _ci95[0, 1],
            "calibration_slope": _b1,
            "slope_se": _se[1],
            "slope_ci_lower": _ci95[1, 0],
            "slope_ci_upper": _ci95[1, 1],
            "n_total": int(y.size),
            "n_events": int(y.sum()),
            "n_clusters": int(pd.unique(grp).size),
            "cov_type": "cluster (patient_id)",
            "n_bins": int(g.shape[0]),
        }])
        write_plot_data(
            plot_data_name or _slugify(ax.get_title() or "calibration_plot"),
            {"curve": curve, "bins": bins, "summary": summary,
             "prediction_histogram": hist},
            out_dir=plot_data_path,
            note=(f"slope={_b1:.4f}, intercept={_b0:+.4f}; "
                  f"cluster-robust SE on patient_id; "
                  f"curve on {len(gx)}-point grid; {g.shape[0]} bins; "
                  f"inset histogram {len(_hist_n)} bins"),
        )

    return fig, fit, g.assign(ci_lo=_lo, ci_hi=_hi)
 


fig, cal_fit, cal_table = plot_calibration_full(dca_df, plot_data=True, plot_data_name="calibration_plot_new")
fig.savefig(PLOT_PATH + 'calibration_plot_new', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Step 4: choose and justify the plausible threshold range ---
dca_range_lo, dca_range_hi = 0.02, 0.20     # <-- the decision; revise after clinical input

_y = dca_df["y"].to_numpy(int)
_p = dca_df["p"].to_numpy(float)
_prev = _y.mean()

_show = [0.02, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]
_rows = []
for _pt in _show:
    _alert = _p >= _pt
    _tp = int((_alert & (_y == 1)).sum())
    _fp = int((_alert & (_y == 0)).sum())
    _fn = int((~_alert & (_y == 1)).sum())
    _ppv = _tp / (_tp + _fp) if (_tp + _fp) else np.nan
    _rows.append({
        "threshold":               _pt,
        "FP tolerated per TP":     _pt / (1 - _pt),
        "alerts per 100 pt-days":  _alert.mean() * 100,
        "sensitivity":             _tp / (_tp + _fn) if (_tp + _fn) else np.nan,
        "PPV":                     _ppv,
        "number needed to alert":  (_tp + _fp) / _tp if _tp else np.nan,
        "net benefit > 0?":        "yes" if _ppv > _pt else "no",
    })

dca_threshold_table = pd.DataFrame(_rows).set_index("threshold")
print(f"prevalence per patient-day = {_prev:.4f}\n")
print(dca_threshold_table.round(3).to_string())

In [ ]:
def dca_net_benefit(y, p, thresholds):
    """Net benefit of the model and of 'treat all' at each threshold.
    NB = TP/n - (FP/n) * pt/(1-pt).  'Treat none' is 0 by definition.
    Note TP and FP are divided by ALL patient-days, not by the alerts."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n, prev = len(y), y.mean()
    nb_model = np.empty(len(thresholds))
    nb_all   = np.empty(len(thresholds))
    for i, pt in enumerate(thresholds):
        w = pt / (1.0 - pt)
        alert = p >= pt
        tp = np.sum(alert & (y == 1)) / n
        fp = np.sum(alert & (y == 0)) / n
        nb_model[i] = tp - fp * w
        nb_all[i]   = prev - (1.0 - prev) * w
    return nb_model, nb_all

dca_thresholds = np.round(np.arange(0.01, 1.01, 0.005), 4)
_nb_model, _nb_all = dca_net_benefit(dca_df["y"], dca_df["p"], dca_thresholds)

dca_curve = pd.DataFrame({
    "threshold":     dca_thresholds,
    "nb_model":      _nb_model,
    "nb_treat_all":  _nb_all,
    "nb_treat_none": 0.0,
})
dca_curve["best_default"] = dca_curve[["nb_treat_all", "nb_treat_none"]].max(axis=1)
dca_curve["nb_gain_vs_best_default"] = dca_curve["nb_model"] - dca_curve["best_default"]

_show = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
print(dca_curve[dca_curve["threshold"].isin(_show)].round(4).to_string(index=False))

_pos = dca_curve.loc[dca_curve["nb_model"] > 0, "threshold"]
_win = dca_curve.loc[dca_curve["nb_gain_vs_best_default"] > 0, "threshold"]
print(f"\nModel NB > 0 for thresholds        {_pos.min():.3f}-{_pos.max():.3f}")
print(f"Model beats BOTH defaults for      {_win.min():.3f}-{_win.max():.3f}")
_in = dca_curve["threshold"].between(dca_range_lo, dca_range_hi)
print(f"Within {dca_range_lo}-{dca_range_hi}: gain over best default "
      f"{dca_curve.loc[_in, 'nb_gain_vs_best_default'].min():+.4f} to "
      f"{dca_curve.loc[_in, 'nb_gain_vs_best_default'].max():+.4f}")

In [ ]:
_w = dca_thresholds / (1 - dca_thresholds)
dca_curve["net_reduction_per_100"] = (dca_curve["nb_model"] - dca_curve["nb_treat_all"]) / _w * 100
dca_curve["alerts_per_100"] = [(dca_df["p"].to_numpy() >= t).mean() * 100 for t in dca_thresholds]

_show = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
print(dca_curve[dca_curve["threshold"].isin(_show)][
    ["threshold", "nb_model", "nb_treat_all", "nb_gain_vs_best_default",
     "alerts_per_100", "net_reduction_per_100"]].round(3).to_string(index=False))

### Standardized decision curve analysis

In [ ]:
def _dca_nb_fast(y, p, th):
    n, prev = len(y), y.mean()
    p1, p0 = np.sort(p[y == 1]), np.sort(p[y == 0])
    tp = (len(p1) - np.searchsorted(p1, th, side="left")) / n
    fp = (len(p0) - np.searchsorted(p0, th, side="left")) / n
    w = th / (1 - th)
    return tp - fp * w, prev - (1 - prev) * w, prev      # <- now also returns prevalence

def dca_bootstrap_snb(df, th, n_boot=2000, seed=42):
    """As before, but each draw is standardised by ITS OWN prevalence."""
    rng = np.random.default_rng(seed)
    _g = {pid: (s["y"].to_numpy(int), s["p"].to_numpy(float))
          for pid, s in df.groupby("patient_id")}
    _pids = np.array(list(_g.keys()))
    m_nb, m_gain  = np.empty((n_boot, len(th))), np.empty((n_boot, len(th)))
    m_snb, m_sgain = np.empty((n_boot, len(th))), np.empty((n_boot, len(th)))
    for b in range(n_boot):
        s = rng.choice(_pids, size=len(_pids), replace=True)
        _yb = np.concatenate([_g[i][0] for i in s])
        _pb = np.concatenate([_g[i][1] for i in s])
        nb, na, prev_b = _dca_nb_fast(_yb, _pb, th)
        gain = nb - np.maximum(na, 0.0)
        m_nb[b], m_gain[b] = nb, gain
        if prev_b > 0:
            m_snb[b], m_sgain[b] = nb / prev_b, gain / prev_b
        else:
            m_snb[b] = m_sgain[b] = np.nan
    return m_nb, m_gain, m_snb, m_sgain

_m_nb, _m_gain, _m_snb, _m_sgain = dca_bootstrap_snb(dca_df, dca_thresholds, n_boot=2000)

dca_prev = dca_df["y"].mean()
dca_curve["snb_model"]      = dca_curve["nb_model"]     / dca_prev
dca_curve["snb_treat_all"]  = dca_curve["nb_treat_all"] / dca_prev
dca_curve["snb_treat_none"] = 0.0
dca_curve["snb_gain_vs_best_default"] = dca_curve["nb_gain_vs_best_default"] / dca_prev

dca_curve["nb_model_lo"]  = np.percentile(_m_nb, 2.5, axis=0)
dca_curve["nb_model_hi"]  = np.percentile(_m_nb, 97.5, axis=0)
dca_curve["gain_lo"]      = np.percentile(_m_gain, 2.5, axis=0)
dca_curve["gain_hi"]      = np.percentile(_m_gain, 97.5, axis=0)
dca_curve["snb_model_lo"] = np.nanpercentile(_m_snb, 2.5, axis=0)
dca_curve["snb_model_hi"] = np.nanpercentile(_m_snb, 97.5, axis=0)
dca_curve["snb_gain_lo"]  = np.nanpercentile(_m_sgain, 2.5, axis=0)
dca_curve["snb_gain_hi"]  = np.nanpercentile(_m_sgain, 97.5, axis=0)
dca_curve["prob_gain_positive"] = (_m_gain > 0).mean(axis=0)

print(f"prevalence = {dca_prev:.4f}  →  max attainable net benefit = {dca_prev:.4f}, sNB caps at 1.0\n")
print(dca_curve[dca_curve["threshold"].isin(_show)][
    ["threshold", "nb_model", "snb_model", "snb_model_lo",
     "snb_model_hi", "snb_treat_all"]].round(3).to_string(index=False))

Standardized net benefit, plotted below.

**Figure 6D** -- Standardized decision curve analysis, external validation cohort

In [ ]:
def _cb_label(x, maxden=100):
    f = Fraction(x / (1 - x)).limit_denominator(maxden)
    return f"{f.numerator}:{f.denominator}"


def dca_plot_snb(curve, label="", show_reduction=False, standardized=True,
                 xlim=(0, 1), aspect=0.66, ylo=-0.5,
                 plot_data=False, plot_data_path=None, plot_data_name=None):
    key  = "snb_model"      if standardized else "nb_model"
    kall = "snb_treat_all"  if standardized else "nb_treat_all"
    klo, khi = ("snb_model_lo", "snb_model_hi") if standardized else ("nb_model_lo", "nb_model_hi")
    ylab = "Standardized net benefit" if standardized else "Net benefit"

    n = 2 if show_reduction else 1
    fig, axes = plt.subplots(n, 1, figsize=(6.6, 6.8) if n == 2 else (6.6, 4.8),
                             sharex=True, gridspec_kw={"height_ratios": [2, 1][:n]})
    axes = np.atleast_1d(axes); ax = axes[0]
    t = curve["threshold"].to_numpy()

    if klo in curve:
        ax.fill_between(t, curve[klo], curve[khi], color=DCA_BLUE, alpha=.18, lw=0, zorder=1)
    ax.plot(t, curve[kall], color=DCA_GRAY, lw=2, ls="--", zorder=2, label="Treat all")
    ax.axhline(0, color=DCA_INK, lw=1.6, ls=":", zorder=2, label="Treat none")
    ax.plot(t, curve[key], color=DCA_BLUE, lw=2.4, zorder=3, label="Model")

    if standardized:
        ax.set_ylim(ylo, 1.06)
    else:
        ax.set_ylim(max(min(0, curve[key].min()) - 0.01, -0.05), curve[key].max() * 1.18)

    ax.set_ylabel(ylab)
    _base = "Standardized decision curve analysis" if standardized else "Decision curve analysis"
    ax.set_title(f"{_base}{' — ' + label if label else ''}", pad=26)
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    ax.grid(axis="y", color="k", alpha=.06, lw=.8); ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    sec = ax.secondary_xaxis("top")
    _tk = [x for x in (1/101, 0.20, 0.40, 0.60, 0.80, 100/101) if xlim[0] <= x <= xlim[1]]
    sec.set_xticks(_tk)
    sec.set_xticklabels([_cb_label(x) for x in _tk], fontsize=8)
    sec.set_xlabel("Cost : benefit ratio", fontsize=8.5, labelpad=4, color=DCA_INK)
    sec.spines["top"].set_visible(False); sec.tick_params(length=3, colors=DCA_INK)

    if show_reduction:
        ax2 = axes[1]
        ax2.axhline(0, color=DCA_INK, lw=1, ls=":", zorder=2)
        ax2.plot(t, curve["net_reduction_per_100"], color=DCA_BLUE, lw=2.4, zorder=3)
        ax2.set_ylabel("Antibiotic starts avoided\nper 100 patient-days\n(vs treat all)")
        ax2.grid(axis="y", color="k", alpha=.06, lw=.8); ax2.set_axisbelow(True)
        ax2.spines[["top", "right"]].set_visible(False)

    axes[-1].set_xlabel("Decision threshold")
    for _a in axes:
        _a.set_xlim(*xlim)

    ax.set_anchor("S"); axes[-1].set_anchor("N")
    ax.set_box_aspect(aspect)
    if n == 2:
        axes[1].set_box_aspect(aspect / 2)
    fig.tight_layout()

    # ---- source data for the journal --------------------------------------
    if plot_data:
        pfx = "snb" if standardized else "nb"
        with np.errstate(divide="ignore", invalid="ignore"):
            _cb = np.where(t < 1, t / (1 - t), np.nan)
        _cb = np.where(np.isfinite(_cb), _cb, np.nan)

        nb = {"threshold": t, "cost_benefit_ratio": _cb,
              f"{pfx}_model": curve[key].to_numpy()}
        if klo in curve:
            nb[f"{pfx}_model_ci_lower"] = curve[klo].to_numpy()
            nb[f"{pfx}_model_ci_upper"] = curve[khi].to_numpy()
        nb[f"{pfx}_treat_all"] = curve[kall].to_numpy()
        nb[f"{pfx}_treat_none"] = np.zeros_like(t, dtype=float)
        tables = {"net_benefit": pd.DataFrame(nb)}

        if show_reduction:
            tables["net_reduction"] = pd.DataFrame({
                "threshold": t,
                "net_reduction_per_100": curve["net_reduction_per_100"].to_numpy(),
            })

        write_plot_data(
            plot_data_name or _slugify(ax.get_title() or f"{pfx}_decision_curve"),
            tables,
            out_dir=plot_data_path,
            note=(f"{ylab}; {'with' if klo in curve else 'no'} 95% CI band; "
                  f"{len(t)} thresholds ({t.min():.4g}-{t.max():.4g}); "
                  f"plotted x-range {xlim[0]}-{xlim[1]}"
                  + (f"; cohort: {label}" if label else "")),
        )

    return fig


fig = dca_plot_snb(dca_curve, label="External cohort", plot_data=True, plot_data_name="Standardized_DCAI")
fig.savefig(PLOT_PATH + 'Standardized_DCAI', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
_show = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
print(f"prevalence = {dca_prev:.4f}\n")
print(dca_curve[dca_curve["threshold"].isin(_show)][
    ["threshold", "snb_model", "snb_model_lo", "snb_model_hi",
     "snb_treat_all", "snb_gain_lo", "snb_gain_hi"]].round(4).to_string(index=False))

### Ind vs. cons

**Supplementary Figure 10A** -- Confusion matrix, induction phase, external validation cohort

In [ ]:
# we can use earlier created eval_df
ind = eval_df.loc[eval_df['sykli_IND']==1]

#df, score, y_pred_mod = custom_score(ind['y_true'], y_prob_test, group_test, threshold=0.29, eval=True)
y_test = ind['infektion_binary']
y_pred_mod = ind['pred_modified_y']
#y_pred_ = ind['original_y']
y_pred_ = (ind['pred_proba'] >= THRESHOLD).astype(int)

# 👉 3. Evaluation metrics
f1 = f1_score(y_test, y_pred_mod)
recall = recall_score(y_test, y_pred_mod)
accuracy = accuracy_score(y_test, y_pred_mod)
precision = precision_score(y_test, y_pred_mod)


# build a metrics table
rows = pd.DataFrame({
    'data':['IND', 'IND_corrected'],
    'F1_score': [f1_score(y_test, y_pred_), f1_score(y_test, y_pred_mod)],
    'kappa': [cohen_kappa_score(y_test, y_pred_), cohen_kappa_score(y_test, y_pred_mod)],
    'recall': [recall_score(y_test, y_pred_), recall_score(y_test, y_pred_mod)],
    'precision' : [precision_score(y_test, y_pred_), precision_score(y_test, y_pred_mod)],
    'accuracy' : [accuracy_score(y_test, y_pred_), accuracy_score(y_test, y_pred_mod)],
    'customf1' : [custom_f1(y_test, y_pred_), custom_f1(y_test, y_pred_mod)],
    'specificity': [recall_score(y_test, y_pred_, pos_label=0), recall_score(y_test, y_pred_mod, pos_label=0)],
    'PNV':[calc_pnv(y_test, y_pred_),calc_pnv(y_test, y_pred_mod)]

})

metrics_df = pd.concat([metrics_df, rows], ignore_index=True)
metrics_df

plot_confusion_matrix_with_percentages(
    y_test, y_pred_, 
    percent_type='row', 
    threshold=THRESHOLD,
    #title='Induction: Confusion Matrix\nTest set',
    title='External validation set, Inductions',
    save_path=PLOT_PATH+'cm_induction_correction_'+str(THRESHOLD).replace('.', '_'),
    cmap=colourmap,
    text_color='white',
    plot_data=True
)

display(metrics_df.style.format({"Value": "{:.4f}"}))


metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_,  THRESHOLD, "External ind")
metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "External ind corrected")
fmt_table(metrics_df_ci)

**Supplementary Figure 10B** -- Confusion matrix, consolidation phase, external validation cohort

In [ ]:
kons = eval_df.loc[eval_df['sykli_IND']!=1]

y_test = kons['infektion_binary']
y_pred_mod = kons['pred_modified_y']
#y_pred_ = kons['original_y']
y_pred_ = (kons['pred_proba'] >= THRESHOLD).astype(int)
# build a metrics table
rows = pd.DataFrame({
    'data':['KONS', 'KONS_corrected'],
    'F1_score': [f1_score(y_test, y_pred_), f1_score(y_test, y_pred_mod)],
    'kappa': [cohen_kappa_score(y_test, y_pred_), cohen_kappa_score(y_test, y_pred_mod)],
    'recall': [recall_score(y_test, y_pred_), recall_score(y_test, y_pred_mod)],
    'precision' : [precision_score(y_test, y_pred_), precision_score(y_test, y_pred_mod)],
    'accuracy' : [accuracy_score(y_test, y_pred_), accuracy_score(y_test, y_pred_mod)],
    'customf1' : [custom_f1(y_test, y_pred_), custom_f1(y_test, y_pred_mod)],
    'specificity': [recall_score(y_test, y_pred_, pos_label=0), recall_score(y_test, y_pred_, pos_label=0)],
    'PNV':[calc_pnv(y_test, y_pred_),calc_pnv(y_test, y_pred_mod)]

})

metrics_df = pd.concat([metrics_df, rows], ignore_index=True)
metrics_df

plot_confusion_matrix_with_percentages(
    y_test, y_pred_, 
    percent_type='row', 
    threshold=THRESHOLD,
    #title='Consolidation: Confusion Matrix\nTest set',
    title='External validation set, Consolidations',
    save_path=PLOT_PATH+'cm_consolidation_correction_'+str(THRESHOLD).replace('.', '_'),
    cmap=colourmap,
    text_color='white',
    plot_data=True
)

display(metrics_df.style.format({"Value": "{:.4f}"}))


metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_,  THRESHOLD, "External kons")
metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "External kons corrected")
fmt_table(metrics_df_ci)

In [ ]:
eval_df.to_csv(DATA_PATH+'misclassified_data_new.csv')